# BKQuiz · 100câu test từ nhóm nguồn giữ riêng: Base và SFT-v2
Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Run all**.
Không chạy OCR/VLM hoặc teacher7B. Giữ CUDA torch của Colab; cài đặt hiện đầy đủ log.
Người dùng tự chạy GPU. Notebook mới không thay notebook/model đang phục vụ.

100câu tham chiếu máy được giữ riêng từ trước train; gồm nhóm trang giữ lại và2tài liệu
không có trong train SFT-v2. CPT cũ đã học27PDF, vì vậy không gọi đây là unseen-CPT.
Không dùng test để chọn checkpoint. Đầu vào generation có source cards chuẩn (oracle context),
không đo retrieval recall. Mặc định chạy **đủ100MCQ và100quiz cho từng bản**; cache theo từng batch.
Đây là bước dài nhất, có thể mất vài giờ trên T4. Batch2mặc định; dùng4nếu GPU đủ RAM.
Muốn chạy nhanh20/50quiz thì đổi GENERATION_LIMIT trước lần đầu và đổi RUN_NAME;
báo cáo luôn giữ đúng mẫu số, không gọi mẫu nhỏ là100quiz đã đánh giá.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,hashlib,zipfile,shutil,stat,json
from pathlib import PurePosixPath

In [ ]:
DRIVE_ROOT=Path('/content/drive/MyDrive/BKQuiz_CPT/SFT_V2_THEORY_20261025')
ZIP_PATH=DRIVE_ROOT/'bkquiz_sft_v2_test100_input.zip'
EXPECTED_SHA256='10510ec4800e9c168873e2b8bbdfe9b06439e708c07c59c633e8a2c88bf19669'
RUN_NAME='final_test_001'
TRAIN_RUN_NAME='sft_v2_001'
BATCH_SIZE=2
GENERATION_LIMIT=100

In [ ]:
def file_sha(path):
    with Path(path).open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()
assert ZIP_PATH.is_file(), str(ZIP_PATH)
LOCAL_ZIP=Path('/content')/ZIP_PATH.name
shutil.copyfile(ZIP_PATH,LOCAL_ZIP)
assert file_sha(LOCAL_ZIP)==EXPECTED_SHA256, 'Notebook vàZIP không cùng bản'
WORK=Path('/content/bkquiz_v2_'+EXPECTED_SHA256[:12]);WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    infos=z.infolist(); names=[i.filename for i in infos]
    assert len(names)==len(set(names)) and len(names)<1000 and sum(i.file_size for i in infos)<512*1024*1024
    for i in infos:
        r=PurePosixPath(i.filename)
        assert not r.is_absolute() and '..' not in r.parts and ':' not in i.filename and chr(92) not in i.filename
        assert stat.S_IFMT(i.external_attr>>16)!=stat.S_IFLNK and i.file_size<128*1024*1024
        assert (WORK/i.filename).resolve().is_relative_to(WORK.resolve())
    m=json.loads(z.read('manifest.json'));assert set(names)=={'manifest.json',*m['files_sha256']}
    z.extractall(WORK)
for name,wanted in m['files_sha256'].items(): assert file_sha(WORK/name)==wanted,name
RUNTIME=WORK/'runtime'
RUN=DRIVE_ROOT/'runs'/RUN_NAME;RUN.mkdir(parents=True,exist_ok=True)

In [ ]:
"""Colab environment bootstrap, learned from the reported missing-pip incident.

Standard library only. Importing this file never installs or deletes anything.
"""
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import uuid

GET_PIP_URL="https://bootstrap.pypa.io/get-pip.py"


def run_logged(command, *, log_path=None, **kwargs):
    """Stream stdout and stderr together; retain the complete error output."""
    if any(key in kwargs for key in ("stdout","stderr","capture_output","text")):
        raise ValueError("run_logged owns output capture")
    kwargs.pop("check",None)
    print("Running:"," ".join(map(str,command)),flush=True)
    process=subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,**kwargs)
    chunks=[]
    with process.stdout:
        for line in process.stdout:
            print(line,end="",flush=True);chunks.append(line)
    code=process.wait();output="".join(chunks)
    if log_path is not None:
        with Path(log_path).open("a",encoding="utf-8") as stream:
            stream.write("COMMAND: "+repr(list(map(str,command)))+"\n"+output+"\nEXIT: "+str(code)+"\n")
    if code:raise subprocess.CalledProcessError(code,command,output=output)
    return subprocess.CompletedProcess(command,code,stdout=output)


def healthy_venv(venv,base_python=sys.executable):
    """An interpreter file alone does not prove that pip or isolation works."""
    venv=Path(venv);python=venv/"bin/python"
    if not python.is_file():return False
    probe="import json,sys;print(json.dumps({'prefix':sys.prefix,'base':sys.base_prefix,'version':list(sys.version_info[:2])}))"
    try:
        own=subprocess.run([str(python),"-c",probe],capture_output=True,text=True,timeout=30)
        base=subprocess.run([str(base_python),"-c",probe],capture_output=True,text=True,timeout=30)
        pip=subprocess.run([str(python),"-m","pip","--version"],capture_output=True,text=True,timeout=30)
        if own.returncode or base.returncode or pip.returncode:return False
        info=json.loads(own.stdout);base_info=json.loads(base.stdout)
        return (Path(info["prefix"]).resolve()==venv.resolve() and info["prefix"]!=info["base"]
                and info["version"]==base_info["version"])
    except (OSError,ValueError,KeyError,subprocess.SubprocessError):return False


def download_get_pip(target):
    """Fetch an official script as a file, not curl|python; record its digest."""
    target=Path(target)
    with urllib.request.urlopen(GET_PIP_URL,timeout=60) as response:
        if not response.geturl().startswith("https://bootstrap.pypa.io/"):
            raise RuntimeError("Unexpected get-pip download destination")
        body=response.read(8*1024*1024+1)
    if not body or len(body)>8*1024*1024:raise RuntimeError("Invalid get-pip download size")
    target.write_bytes(body)
    return hashlib.sha256(body).hexdigest()


def ensure_colab_venv(venv, *, allowed_root=Path("/content"), base_python=sys.executable,
                      allow_system_packages=True, runner=run_logged, downloader=download_get_pip):
    """Repair only this disposable venv; never touch Drive/checkpoints/data."""
    requested=Path(venv);root=Path(allowed_root).resolve();venv=requested.resolve()
    if requested.is_symlink() or venv.parent!=root or venv==root or venv.name in {"drive","datasets","runs","checkpoints"}:
        raise ValueError("Venv must be a dedicated direct child of the allowed scratch directory")
    if healthy_venv(venv,base_python):
        print("Reusing healthy Python + pip:",venv,flush=True)
        return venv/"bin/python"
    if venv.exists():
        if not venv.is_dir() or (any(venv.iterdir()) and not (venv/"pyvenv.cfg").is_file()):
            raise ValueError("Existing path is not a recognizable disposable venv; choose a new venv path")
        # Keep the old environment recoverable. No recursive data deletion.
        backup=venv.with_name(venv.name+".broken."+uuid.uuid4().hex[:12])
        if backup.resolve().parent!=root:raise ValueError("Invalid venv backup target")
        venv.rename(backup);print("Old broken venv retained:",backup,flush=True)
    create=[str(base_python),"-m","venv","--system-site-packages","--without-pip",str(venv)]
    log_path=root/(venv.name+"_setup.log")
    try:runner(create,log_path=log_path)
    except subprocess.CalledProcessError as error:
        details=str(error.output or "")+str(error.stderr or "")
        missing_venv="No module named venv" in details or "No module named 'venv'" in details
        if not missing_venv or not allow_system_packages or not shutil.which("apt-get"):raise
        # Generic python3-venv may target a different interpreter than Colab's.
        version=subprocess.run([str(base_python),"-c","import sys;print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
                               check=True,capture_output=True,text=True).stdout.strip()
        if not __import__('re').fullmatch(r"3\.\d+",version):raise RuntimeError("Cannot identify active Python version")
        runner(["apt-get","update"],log_path=log_path)
        runner(["apt-get","install","-y","python"+version+"-venv"],log_path=log_path)
        runner(create,log_path=log_path)
    python=venv/"bin/python"
    if not python.is_file():raise RuntimeError("Venv creation produced no Python executable; inspect setup log")
    script=root/(venv.name+"_get-pip.py")
    script_sha=downloader(script)
    runner([str(python),str(script)],log_path=log_path)
    if not healthy_venv(venv,base_python):raise RuntimeError("Python/pip health check still failed; stop and inspect setup log")
    (venv/"colab_bootstrap.json").write_text(json.dumps({"base_python":str(base_python),"method":"without-pip_then_official_get-pip",
          "get_pip_url":GET_PIP_URL,"get_pip_sha256":script_sha},indent=2),encoding="utf-8")
    return python

VENV=Path('/content/bkquiz_sft_v2_venv')
PYTHON=ensure_colab_venv(VENV)
(RUNTIME/'sitecustomize.py').write_text("import importlib.util\n_original=importlib.util.find_spec\ndef _find(name,*args,**kwargs):\n    if name.split('.')[0] in {'torchvision','torchaudio','librosa'}:return None\n    return _original(name,*args,**kwargs)\nimportlib.util.find_spec=_find\n",encoding='utf-8')
ENV=dict(os.environ,PYTHONPATH=str(RUNTIME),TOKENIZERS_PARALLELISM='false')
run_logged([PYTHON,'-m','pip','install','-r',RUNTIME/'requirements.txt'],env=ENV,log_path=RUN/'setup.log')
run_logged([PYTHON,'-c','import torch; assert torch.cuda.is_available(), "Select GPU runtime"; print(torch.__version__,torch.cuda.get_device_properties(0))'],env=ENV,log_path=RUN/'setup.log')
def launch(script,*args):
    run_logged([PYTHON,'-u',RUNTIME/script,*map(str,args)],env=ENV,cwd=RUNTIME,log_path=RUN/'execution.log')

In [ ]:
SFT_RUN=DRIVE_ROOT/'runs'/TRAIN_RUN_NAME/'train'
assert (SFT_RUN/'completed.json').is_file(), 'Chạy notebook02 trước hoặc sửa TRAIN_RUN_NAME cho đúng'
launch('evaluate_v2.py','--root',WORK,'--mode','preflight')
for variant in ('base','sft_v2'):
    launch('evaluate_v2.py','--root',WORK,'--output',RUN/'evaluation','--sft-run',SFT_RUN,
           '--variant',variant,'--batch-size',BATCH_SIZE,'--generation-limit',GENERATION_LIMIT)
launch('evaluate_v2.py','--root',WORK,'--output',RUN/'evaluation','--mode','report')
OUTPUT=RUN/'evaluation/sft_v2_final_test_results.zip'
from IPython.display import Markdown,display
display(Markdown((RUN/'evaluation/FINAL_TEST.md').read_text(encoding='utf-8')))

In [ ]:
from google.colab import files
assert OUTPUT.is_file(), 'Chưa có kết quả hoàn chỉnh'
print('Lưu trên Drive:',OUTPUT)
files.download(str(OUTPUT))

Giữ RUN_NAME để tiếp tục cache/checkpoint cùng cấu hình. Khi đổi code, dataset, thư viện,
batch hoặc số mẫu, dùng RUN_NAME mới. Nếu chưa có checkpoint đầu tiên, cần run mới để train lại từ đầu.
Các cờ human_verified/expert_semantically_verified vẫn false nếu chưa có lượt duyệt thật.